# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library. All record sets, fields, and columns are referenced by their `@id` identifiers for full reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL and can be explored using the FAIR metadata principles.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # .to_json() is NOT called; keep as object
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields/columns, and all accessible IDs for robust referencing.

**Note:** All discovered record sets, fields and columns are referenced via their `@id` values.

In [ ]:
# List all record sets declared by the dataset, with their @id
record_sets = list(dataset.record_sets)
print(f"Number of Record Sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '(no name)')}")
    print("  Fields & Columns:")
    # For each field in the record set, print the @id, label and dataType
    for field in rs.get('field', []):
        print(f"    - Field @id: {field['@id']}, name: {field.get('name', '')}, dataType: {field.get('dataType', '')}")
    # For each column in the record set
    for col in rs.get('column', []):
        print(f"    - Column @id: {col['@id']}, name: {col.get('name', '')}, dataType: {col.get('dataType', '')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All referencing is done via `@id` fields as output in the previous cell.

**Note:** For this dataset, the main table containing experimental results is typically in the record set whose name or `@id` resembles "main", "dataset", or refers to the mass spectrometry outputs. Replace the variable `selected_record_set_id` below by the main record set's `@id` printed above.

In [ ]:
# Configure the main record set @id from above
selected_record_set_id = None
# If only one record set is present, use it automatically
if len(record_sets) == 1:
    selected_record_set_id = record_sets[0]['@id']
else:
    # Otherwise, set this manually to e.g. record_sets[0]['@id']
    # selected_record_set_id = '<paste-record-set-@id-here>'
    selected_record_set_id = record_sets[0]['@id']  # For this notebook, pick the first

print(f"Using Record Set @id: {selected_record_set_id}\n")

# Load all records from the specified record set
records = list(dataset.records(record_set=selected_record_set_id))
df = pd.DataFrame(records)
print(f"Loaded {len(df)} records from Record Set: {selected_record_set_id}")
print(f"Columns (@id or label): {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping by relevant keys.

Below, select a quantitative (numeric) field using its exact `@id` as output above, and a grouping field for aggregation.

In [ ]:
# --- CONFIGURE FIELD IDS HERE ---
# Choose a numeric field (e.g. percent_coverage, total_peptides, MW)
numeric_field_id = None
# Pick the first detected numeric column containing 'coverage' or 'MW' or 'count'
for field in df.columns:
    if 'coverage' in field.lower() or 'mw' == field.lower() or 'abund' in field.lower() or 'peptide' in field.lower():
        numeric_field_id = field
        break
if not numeric_field_id:
    # Fallback to first column
    numeric_field_id = df.columns[0]
print(f"Using numeric field: {numeric_field_id}")

# Choose a grouping (categorical) field
group_field_id = None
for field in df.columns:
    # Use accession, description or sample
    if 'sample' in field.lower() or 'description' in field.lower() or 'accession' in field.lower():
        group_field_id = field
        break
filtered_df = df.copy()

# Ensure numeric field is float type
filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
threshold = filtered_df[numeric_field_id].quantile(0.75)  # Top 25%
filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

if group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. The following example creates a histogram and, if a group field exists, a boxplot.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

if group_field_id is not None and group_field_id in filtered_df.columns:
    # Display boxplot group comparison if unique group values are reasonable
    group_count = filtered_df[group_field_id].nunique()
    if group_count > 1 and group_count < 25:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=30, ha='right')
        plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded and summarized the structure of a FAIR Croissant mass spectrometry dataset using `mlcroissant`.
- Explored record sets and referenced all fields and columns explicitly by `@id`.
- Loaded records into pandas DataFrames for flexible analysis.
- Filtered, normalized and grouped data on quantitative fields (e.g., protein abundance, percent coverage, or peptide counts).
- Produced visualizations to illustrate the main distributions in the dataset.

For more detailed analyses, refer to the dataset's Croissant schema and repeat the same approach on any record set or field using its `@id`.